In [1]:
import torch, torchvision, torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np, random

# Reproducibility
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2023, 0.1994, 0.2010)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)


100%|██████████| 170M/170M [01:17<00:00, 2.20MB/s] 


In [6]:
import torch.nn.functional as F
import random

def _rand_bbox(W, H, lam):
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, x2 = np.clip(cx - cut_w // 2, 0, W), np.clip(cx + cut_w // 2, 0, W)
    y1, y2 = np.clip(cy - cut_h // 2, 0, H), np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def mixup_cutmix_collate(alpha_mix=0.2, alpha_cut=1.0, p_mix=0.5):
    def collate(batch):
        imgs, labels = zip(*batch)
        x = torch.stack(imgs)
        y = torch.tensor(labels)
        if random.random() < p_mix:
            # MixUp
            lam = np.random.beta(alpha_mix, alpha_mix)
            index = torch.randperm(x.size(0))
            x = lam * x + (1 - lam) * x[index]
            return x, (y, y[index], lam, "mixup")
        else:
            # CutMix
            lam = np.random.beta(alpha_cut, alpha_cut)
            index = torch.randperm(x.size(0))
            W, H = x.size(3), x.size(2)
            x1, y1, x2, y2 = _rand_bbox(W, H, lam)
            x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
            lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
            return x, (y, y[index], lam, "cutmix")
    return collate

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                          num_workers=0, pin_memory=False,
                          collate_fn=mixup_cutmix_collate())
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False,
                          num_workers=0, pin_memory=False)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = resnet18(weights=None, num_classes=10)
# optional dropout injection:
# model.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(model.fc.in_features, 10))
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.02)

# Warmup + cosine
def lr_lambda(epoch, warmup=5, total=200):
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / (total - warmup)
    return 0.5 * (1 + np.cos(np.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda e: lr_lambda(e))

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


In [9]:
def train_epoch():
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y_pack in train_loader:
        x = x.to(device)
        y1, y2, lam, kind = y_pack
        y1 = y1.to(device); y2 = y2.to(device)

        optimizer.zero_grad()
        logits = model(x)
        # mixed targets
        loss = lam * criterion(logits, y1) + (1 - lam) * criterion(logits, y2)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = logits.argmax(1)
            # for accuracy, use primary y1 as reference (approx)
            correct += (preds == y1).sum().item()
            total += y1.size(0)
            loss_sum += loss.item() * y1.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch():
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    all_preds, all_probs, all_labels = [], [], []
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        loss_sum += loss.item() * y.size(0)

        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())
        all_labels.append(y.cpu())
    return (loss_sum / total, correct / total,
            torch.cat(all_preds), torch.cat(all_probs), torch.cat(all_labels))

for epoch in range(1):
    tr_loss, tr_acc = train_epoch()
    te_loss, te_acc, preds, probs, labels = eval_epoch()
    scheduler.step()
    print(f"Epoch {epoch:03d} | tr_loss {tr_loss:.4f} acc {tr_acc:.4f} | te_loss {te_loss:.4f} acc {te_acc:.4f}")


Epoch 000 | tr_loss 1.9489 acc 0.2713 | te_loss 1.6242 acc 0.4741


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def compute_metrics(preds, probs, labels, num_classes=10):
    y_true = labels.numpy()
    y_pred = preds.numpy()
    y_prob = probs.numpy()  # shape [N, C]

    # Macro P/R/F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    # Macro AUC: one-vs-rest
    y_true_ovr = np.eye(num_classes)[y_true]
    auc_macro = roc_auc_score(y_true_ovr, y_prob, average='macro', multi_class='ovr')

    acc = (y_true == y_pred).mean()
    return acc, precision, recall, f1, auc_macro

acc, P, R, F1, AUC = compute_metrics(preds, probs, labels)
print(f"Test Acc={acc:.4f}  P={P:.4f}  R={R:.4f}  F1={F1:.4f}  AUC={AUC:.4f}")
